# Phase 1: Unified Data Preprocessing
Step 1: Load and Concatenate

In [ ]:
import pandas as pd
import numpy as np

# Load datasets
train_df = pd.read_csv('../data/dataset/train.csv')
test_df = pd.read_csv('../data/dataset/test.csv')

# Add split column
train_df['split'] = 'train'
test_df['split'] = 'test'

# Fill missing demand in test with 0.0 temporarily
test_df['demand'] = 0.0

# Concatenate
full_df = pd.concat([train_df, test_df], ignore_index=True)

# Display basic info
print(full_df.info())
display(full_df.head())
display(full_df.tail())

### Step 2: Impute Missing Values
Handling missing values in RoadType, Weather, and Temperature based on spatial and temporal rules.

In [ ]:
# 1. RoadType Imputation
# Mode per geohash
roadtype_mode = full_df.groupby('geohash')['RoadType'].transform(lambda x: x.mode()[0] if not x.mode().empty else np.nan)
full_df['RoadType'] = full_df['RoadType'].fillna(roadtype_mode)
full_df['RoadType'] = full_df['RoadType'].fillna('Unknown')

# 2. Weather Imputation
# Sort full_df by ['geohash', 'day', 'timestamp']
full_df = full_df.sort_values(by=['geohash', 'day', 'timestamp'])
# Forward-fill grouped by geohash
full_df['Weather'] = full_df.groupby('geohash')['Weather'].ffill()
# Fill remaining NaNs with 'Clear'
full_df['Weather'] = full_df['Weather'].fillna('Clear')

# 3. Temperature Imputation
# Group by geohash and apply linear interpolation
full_df['Temperature'] = full_df.groupby('geohash')['Temperature'].transform(lambda x: x.interpolate(method='linear'))
# Fill remaining NaNs with the global median Temperature for that day
day_median_temp = full_df.groupby('day')['Temperature'].transform('median')
full_df['Temperature'] = full_df['Temperature'].fillna(day_median_temp)
# If still any NaN (e.g., no temperature for the whole day), fill with global median
full_df['Temperature'] = full_df['Temperature'].fillna(full_df['Temperature'].median())

# Verify imputation
print("Missing values after imputation:")
print(full_df[['RoadType', 'Weather', 'Temperature']].isnull().sum())
